# Run Comparative Judging

This notebook demonstrates how to run the Comparative Judging system on remote viewing sessions fetched from the Social RV API.

## How Comparative Judging Works (Social RV's Multi-Pass Elimination)

This implementation uses the **exact same logic** as Social RV's production system:

1. **Fetch Session Data**: Get a session from the API along with its target and decoys
2. **Prepare Inputs**: Create session file and target objects with URLs
3. **Multi-Pass Elimination**: The AI performs up to 3 rounds of judging:
   - **Pass 1**: 10 targets → select top 3 (ranks 1-3)
   - **Pass 2**: Remaining 7 → select top 3 (ranks 4-6)
   - **Pass 3**: Remaining 4 → select top 3 (ranks 7-9)
   - **Final**: Last remaining target gets rank 10
4. **Early Stopping**: If the correct target is found in any pass, judging stops

## Requirements

- OpenAI API key (for GPT-4o)
- Social RV Research API key
- Node.js and npm (for running the TypeScript implementation)


In [ ]:
# Setup
import sys
sys.path.insert(0, '../src')

from dotenv import load_dotenv
load_dotenv('../.env')

from comparative_judging import (
    SocialRVClient,
    SessionFile,
    Target,
    perform_comparative_judging,
    install_nodejs_dependencies,
    create_session_file_from_url,
    create_target_from_url,
)

# Initialize the API client
client = SocialRVClient()
print(f"✅ Client initialized")


## Install Node.js Dependencies

The comparative judging system uses the exact TypeScript code from Social RV.
We need to install its dependencies first.


In [ ]:
# Install Node.js dependencies (only needs to be run once)
print("📦 Installing Node.js dependencies...")
success = install_nodejs_dependencies()
if success:
    print("✅ Dependencies installed successfully")
else:
    print("⚠️  Failed to install dependencies. You may need to run 'npm install' manually in the nodejs_wrapper directory.")


## Find a Session with Decoys

First, let's find a session that has already been through comparative judging (has decoy IDs).


In [ ]:
# Fetch sessions and find one with decoys
print("📥 Fetching sessions to find one with decoys...")

sessions = client.fetch_all_sessions(
    include_low_value=False,
    max_sessions=100
)

# Find sessions with decoys (need exactly 9 decoys for the multi-pass system)
sessions_with_decoys = [s for s in sessions if s.decoy_ids and len(s.decoy_ids) == 9]

print(f"\n✅ Found {len(sessions_with_decoys)} sessions with 9 decoys")

if sessions_with_decoys:
    # Pick a session with session media
    sessions_with_media = [s for s in sessions_with_decoys if s.session_media_urls]
    sample_session = sessions_with_media[0] if sessions_with_media else sessions_with_decoys[0]
    
    print(f"\n📋 Selected session:")
    print(f"   ID: {sample_session.id}")
    print(f"   User: {sample_session.user_display_name}")
    print(f"   Target Coordinate: {sample_session.target_coordinate}")
    print(f"   Existing CJ Rank: {sample_session.comparative_judging_rank}")
    print(f"   Decoys: {len(sample_session.decoy_ids)}")
    print(f"   Session Media: {len(sample_session.session_media_urls)} files")
else:
    print("❌ No sessions with 9 decoys found")


## Fetch Full Session Data

Now let's fetch the complete session data including the target and all decoys.


In [ ]:
# Fetch full session data with target and decoys
print("📥 Fetching full session data...")

full_data = client.get_session_with_decoys(sample_session.id)

session = full_data['session']
target = full_data['target']
decoys = full_data['decoys']

print(f"\n🎯 Correct Target:")
print(f"   ID: {target.id}")
print(f"   Description: {target.description[:100]}...")
print(f"   Image URL: {'Yes' if target.image_url else 'No'}")

print(f"\n🎭 Decoys ({len(decoys)}):")
for i, decoy in enumerate(decoys[:5]):
    print(f"   {i+1}. {decoy.id[:8]}... - {decoy.description[:50]}...")
if len(decoys) > 5:
    print(f"   ... and {len(decoys) - 5} more")

print(f"\n📁 Session Media ({len(session.session_media_urls)} files):")
for i, media in enumerate(session.session_media_urls[:3]):
    print(f"   {i+1}. {media.get('mime_type', 'unknown')}")


## Prepare Inputs for the Judge

Create SessionFile and Target objects with URLs (no encoding needed - the TypeScript code handles downloading).


In [ ]:
# Create session files from URLs
print("📥 Creating session file objects...")

session_files = []
for i, media in enumerate(session.session_media_urls):
    url = media.get('url')
    mime_type = media.get('mime_type', '')
    
    if url:
        try:
            session_file = create_session_file_from_url(url, f"session_{i+1}")
            session_files.append(session_file)
            print(f"   ✅ Created session file {i+1}: {mime_type}")
        except Exception as e:
            print(f"   ❌ Failed to create session file {i+1}: {e}")

print(f"\n📁 Prepared {len(session_files)} session files")

if not session_files:
    raise ValueError("❌ No session files available")


In [ ]:
# Create target objects from URLs
print("📥 Creating target objects...")

# Create correct target
print(f"\n🎯 Creating correct target...")
if not target.image_url:
    raise ValueError("❌ Target has no image URL - cannot proceed with comparative judging")

target_obj = create_target_from_url(
    target_id=target.id,
    description=target.description,
    image_url=target.image_url
)
print(f"   ✅ Created target: {target.id[:8]}...")

# Create all decoys
print(f"\n🎭 Creating decoy objects...")
decoy_objs = []

for i, decoy in enumerate(decoys):
    if not decoy.image_url:
        print(f"   ⚠️  Skipping decoy {i+1}: no image URL")
        continue
    try:
        decoy_obj = create_target_from_url(
            target_id=decoy.id,
            description=decoy.description,
            image_url=decoy.image_url
        )
        decoy_objs.append(decoy_obj)
        print(f"   ✅ Created decoy {i+1}/{len(decoys)}: {decoy.id[:8]}...")
    except Exception as e:
        print(f"   ❌ Failed to create decoy {i+1}: {e}")

print(f"\n✅ Prepared 1 target + {len(decoy_objs)} decoys")

if len(decoy_objs) != 9:
    print(f"\n⚠️  WARNING: Expected 9 decoys but got {len(decoy_objs)}")
    print("   The multi-pass elimination expects exactly 9 decoys (10 total targets)")


## Run the Comparative Judge

Now we'll run the AI judge using Social RV's exact multi-pass elimination logic.

⚠️ **Note**: This will make API calls to OpenAI (GPT-4o), which has associated costs.
The multi-pass system may make up to 3 API calls (but stops early if the correct target is found).


In [ ]:
# Validate inputs before running
if not session_files:
    raise ValueError("❌ Cannot run comparative judging: No session files available.")

if not decoy_objs:
    raise ValueError("❌ Cannot run comparative judging: No decoy objects were created.")

# Run the comparative judge
print("🧠 Running Comparative Judge (Social RV Multi-Pass Logic)...")
print(f"   Session files: {len(session_files)}")
print(f"   Target + Decoys: 1 + {len(decoy_objs)} = {1 + len(decoy_objs)} total")
print("\n   This may take 30-90 seconds (up to 3 passes)...\n")

# Run the judge
result = perform_comparative_judging(
    session_files=session_files,
    current_target=target_obj,
    historical_targets=decoy_objs
)

print("✅ Judging complete!")


## Results


In [ ]:
# Display results
print("=" * 60)
print("📊 COMPARATIVE JUDGING RESULTS")
print("=" * 60)

print(f"\n🎯 Correct Target Rank: {result.correct_target_rank} out of {result.total_targets_ranked}")

# Compare with existing rank
if session.comparative_judging_rank:
    print(f"\n📈 Comparison with existing CJ rank:")
    print(f"   Original (Social RV): {session.comparative_judging_rank}")
    print(f"   New (This run):       {result.correct_target_rank}")
    if result.correct_target_rank == session.comparative_judging_rank:
        print(f"   ✅ MATCH! Results are identical.")
    else:
        print(f"   ⚠️  Different results (this is expected due to randomization)")

print(f"\n📝 Overall Reasoning:")
print(f"   {result.overall_reasoning[:500]}...")

print(f"\n🏆 Rankings (stopped at rank {result.total_targets_ranked}):")
for match in sorted(result.top_matches, key=lambda x: x.rank):
    is_correct = match.target_id == target.id
    marker = "🎯" if is_correct else "  "
    print(f"   {marker} Rank {match.rank}: {match.target_id[:12]}...")
    print(f"       {match.reasoning[:100]}...")


## Understanding the Results

### How the Multi-Pass System Works

1. **Pass 1**: AI sees all 10 targets and picks top 3 (ranks 1-3)
   - If correct target is found → STOP
2. **Pass 2**: AI sees remaining 7 targets and picks top 3 (ranks 4-6)
   - If correct target is found → STOP
3. **Pass 3**: AI sees remaining 4 targets and picks top 3 (ranks 7-9)
   - If correct target is found → STOP
4. **Final**: Last remaining target automatically gets rank 10

### Key Metrics

- **Correct Target Rank**: Where the actual target ranked (1 = best match, 10 = worst)
- **Total Targets Ranked**: How many targets were ranked before stopping (3, 6, 9, or 10)
- **Early Stopping**: If the system stopped before ranking all 10, it found the correct target

### Interpreting Rankings

- **Rank 1**: The AI thinks this target best matches the session data
- A session is considered "successful" if the correct target ranks 1st
- Random chance with 10 targets would give an average rank of 5.5
- The multi-pass system is designed to be more efficient than ranking all 10 at once

### Why Results May Differ from Social RV

Even though this uses the **exact same code**, results may differ because:
1. **Random shuffling**: Targets are shuffled before each run
2. **AI non-determinism**: Even with low temperature, GPT-4o has some randomness
3. **Different decoys**: If you're testing with different decoys than the original session

### Next Steps

- Run this on multiple sessions to analyze performance
- Compare results with the original CJ ranks from the platform
- Analyze how often the system stops early (finds correct target in first 3, 6, or 9)
- Study the reasoning to understand what the AI is looking for
